# SG-VO — LoRA Online Adaptation Experiment
### ICRA Research Notebook

**Before running:** `Runtime → Change runtime type → T4 GPU`

Runs four conditions and compares them in a results table:

| # | Condition | Params updated |
|---|---|---|
| A | Offline VO (no adaptation) | 0 |
| B | Full online adaptation (paper) | 39M |
| C | LoRA-attention rank 8 | 21K (0.08%) |
| D | LoRA-attention + trigger 0.8 | 21K, ~50% of frames |

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone Repo and Install Dependencies ──────────────────────────────
import os
if not os.path.exists('/content/SG-VO'):
    !git clone https://github.com/AreebaAzizRajput/SG-VO.git /content/SG-VO
else:
    %cd /content/SG-VO
    !git pull
%cd /content/SG-VO
!pip install -q einops path imageio blessings progressbar2 scikit-image tqdm gdown
print('✅ Done')

In [ ]:
# ── Cell 3: Download Pretrained Checkpoints ──────────────────────────────────
import os, glob, shutil
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/exp_pose112_model_best.pth.tar'):
    print('Downloading pretrained models from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1twZsg2pxMLwcUUnFszBn1ryaEtuoEfs3' \
        -O checkpoints/ --quiet
    for f in glob.glob('checkpoints/**/*.pth.tar', recursive=True):
        dest = os.path.join('checkpoints', os.path.basename(f))
        if f != dest: shutil.move(f, dest)

!ls -lh checkpoints/*.pth.tar

In [ ]:
# ── Cell 4: Download Camera Intrinsics ───────────────────────────────────────
import os
os.makedirs('data/kitti_odom/sequences', exist_ok=True)

if not os.path.exists('data/kitti_odom/sequences/kitti_odom256_intrinsics/cam_09.txt'):
    print('Downloading camera intrinsics...')
    !gdown --folder 'https://drive.google.com/drive/folders/1n81QDHaG3lIxnxybl9I6knPHxPngTsD8' \
        -O data/kitti_odom/sequences/ --quiet

!ls data/kitti_odom/sequences/kitti_odom256_intrinsics/

In [ ]:
# ── Cell 5: Download KITTI Odometry Sequences (09 & 10) from GitHub Release ──
# Seq 09/10 image_2 frames (~1.5 GB total) are hosted as release assets on this
# repo (tag: kitti-data-v1), pre-packaged once via remotezip from the official
# KITTI odometry color zip. A plain wget of two tars replaces the old per-frame
# range-request fetch — a fresh Colab session is data-ready in ~2 minutes.
import os, glob, shutil, subprocess

DATASET_DIR = 'data/kitti_odom/sequences'
os.makedirs(DATASET_DIR, exist_ok=True)
for d in ['vo_results_offline','vo_results_full','vo_results_lora','vo_results_lora_trigger']:
    os.makedirs(d, exist_ok=True)

RELEASE = 'https://github.com/AreebaAzizRajput/SG-VO/releases/download/kitti-data-v1'

def seq_ok(seq):
    return len(glob.glob(f'{DATASET_DIR}/{seq}/image_2/*.png')) > 100

for seq in ['09', '10']:
    if seq_ok(seq):
        print(f'✅ Seq {seq} already present — skipping')
        continue
    tar = f'kitti_odom_{seq}.tar'
    print(f'Downloading {tar} ...')
    subprocess.run(['wget', '-q', '--show-progress', f'{RELEASE}/{tar}', '-O', f'/content/{tar}'], check=True)
    # Tar contains <seq>/image_2/*.png — extract straight into the sequences dir
    subprocess.run(['tar', '-xf', f'/content/{tar}', '-C', DATASET_DIR], check=True)
    os.remove(f'/content/{tar}')

# Final verification
SEQUENCES = [s for s in ['09','10'] if seq_ok(s)]
print(f'\nAvailable sequences: {SEQUENCES}')
for s in ['09','10']:
    n = len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png'))
    flag = '✅' if n > 100 else '❌'
    print(f'  Seq {s}: {n} images {flag}')
_,_,free = shutil.disk_usage('/'); print(f'Free disk: {free/1e9:.0f} GB')

In [ ]:
# ── Cell 6: Install cam.txt Intrinsics ───────────────────────────────────────
import os, shutil, glob

INTR_DIR    = 'data/kitti_odom/sequences/kitti_odom256_intrinsics'
DATASET_DIR = 'data/kitti_odom/sequences'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    src  = f'{INTR_DIR}/cam_{seq}.txt'
    dest = f'{DATASET_DIR}/{seq}/image_2/cam.txt'
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f'✅ cam.txt → seq {seq}')
    else:
        print(f'⚠️  Missing intrinsics for seq {seq}: {src}')

In [ ]:
# ── Cell 7: LoRA Sanity Check ────────────────────────────────────────────────
# PoseResNet ALWAYS uses ResNet-18 regardless of --resnet-layers flag.
# (--resnet-layers only controls DispResNet, the depth network.)
# Using num_layers=50 here caused a channel mismatch: ResNet-50 outputs
# 2048-channel features but crossAttention expects 512.
import sys, torch
sys.path.insert(0, '/content/SG-VO')
from lora import inject_lora_pose_net, freeze_all, count_lora_params
from models import PoseResNet

pose_net = PoseResNet(num_layers=18, pretrained=False)  # always 18 for pose
n_total  = sum(p.numel() for p in pose_net.parameters())
freeze_all(pose_net)
inject_lora_pose_net(pose_net, rank=8, alpha=1.0, targets='attention')
n_lora = count_lora_params(pose_net)
print(f'LoRA params: {n_lora:,} / {n_total:,}  ({100*n_lora/n_total:.3f}%)')
print(f'Expected:    21,504 / 25,735,766  (0.084%)')

with torch.no_grad():
    out = pose_net(torch.randn(1, 36, 256, 832), torch.randn(1, 36, 256, 832))
print(f'Forward pass OK — output shape: {out.shape}')
print('✅ LoRA ready')

In [ ]:
# ── Cell 7b: Diagnostic — verify experiments will produce results ────────────
# Run this AFTER Cell 6 (cam.txt) and BEFORE the four experiment conditions.
# It runs Condition A (offline VO) directly with stderr captured, so any crash
# (ImportError, checkpoint mismatch, missing cam.txt) shows up here in ~2 min
# instead of being swallowed by the notebook's !python cells.
import subprocess, glob, os

# 1. Confirm KITTI data survived the session
for s in ['09', '10']:
    n = len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png'))
    print(f'Seq {s}: {n} frames  ({"OK" if n > 100 else "MISSING — re-run Cell 5"})')

# 2. Run offline VO directly with stderr captured (Condition A)
print('\n=== Running test_vo.py on Seq 09 (diagnostic) ===')
r = subprocess.run(
    ['python', 'test_vo.py',
     '--img-height', '256', '--img-width', '832', '--sequence', '09',
     '--pretrained-posenet', 'checkpoints/exp_pose112_model_best.pth.tar',
     '--dataset-dir', 'data/kitti_odom/sequences/',
     '--output-dir', 'vo_results_offline/'],
    capture_output=True, text=True)

print('\n--- return code:', r.returncode)
print('--- STDOUT (tail) ---\n', r.stdout[-2000:])
print('--- STDERR (tail) ---\n', r.stderr[-2000:])
wrote = os.path.exists('vo_results_offline/09.txt')
print('--- 09.txt written? ', wrote)
print('\n', '✅ Results are being written — proceed to the experiment cells.'
      if wrote else
      '❌ No results. Read STDERR above; do NOT run the long cells until this is fixed.')


## Condition A — Offline VO (baseline, no adaptation)

In [ ]:
# ── Cell 8: Offline VO ───────────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]
print(f'Running on sequences: {SEQUENCES}')

for seq in SEQUENCES:
    print(f'\n=== Offline VO — Seq {seq} ===')
    !python test_vo.py \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir vo_results_offline/

print('\n=== Offline results ===')
!python kitti_eval/eval_odom.py --result=vo_results_offline/ --align=7dof

## Condition B — Full Online Adaptation (paper baseline)

In [ ]:
# ── Cell 9: Full Online VO ───────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== Full Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_full/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border

print('\n=== Full online results ===')
!python kitti_eval/eval_odom.py --result=vo_results_full/ --align=7dof

## Condition C — LoRA-Attention Adaptation (new contribution)

In [ ]:
# ── Cell 10: LoRA Online VO ──────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention

print('\n=== LoRA results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora/ --align=7dof

## Condition D — LoRA + Trigger (efficiency variant)

In [ ]:
# ── Cell 11: LoRA + Trigger ──────────────────────────────────────────────────
# Trigger probe uses the contrast-invariant RATIO signal (predicted-pose loss /
# zero-motion loss) so the same trigger config works on vKITTI fog, where the
# raw photometric signal is confounded (fog LOWERS loss). In-domain the CUSUM
# still calibrates from the first 100 frames. The photo-signal result (D-v3,
# 2026-07-15: 3.63/4.84 t_err, 99.8% skipped) stays in results_backup for
# comparison; this rerun makes D's signal consistent across all experiments.
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA + Trigger — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora_trigger/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention \
        --cusum-h 16 \
        --probe-signal ratio

print('\n=== LoRA + Trigger results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora_trigger/ --align=7dof

### Probe-only loss recording — trigger design tool
Records the per-frame photometric loss **without any adaptation** so trigger
designs (spike / dual-EMA / CUSUM) can be simulated offline on real traces
before spending GPU time. Download lands as `probe_losses.zip`.

In [ ]:
# ── Cell 11b: Probe-only loss trace recording (no adaptation) ────────────────
import glob, zipfile, os
PROBE_SIGNAL = 'ratio'   # 'photo' (legacy, contrast-confounded) | 'ratio' | 'bnstats'
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
os.makedirs('vo_results_probe', exist_ok=True)
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'{DATASET_DIR}{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== Probe-only loss recording — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_probe/ \
        --sequence-length 3 \
        --resnet-layers 50 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --probe-only \
        --probe-signal {PROBE_SIGNAL}

with zipfile.ZipFile(f'/content/probe_losses_{PROBE_SIGNAL}.zip', 'w') as z:
    for f in glob.glob('vo_results_probe/probe_losses_*.txt'):
        z.write(f)
from google.colab import files
files.download(f'/content/probe_losses_{PROBE_SIGNAL}.zip')

## Results Comparison Table

In [ ]:
# ── Cell 12: Comparison Table ────────────────────────────────────────────────
import subprocess, re, os

PAPER = {
    'offline': {'09': (7.08, 2.48), '10': (8.72, 3.11)},
    'online':  {'09': (5.21, 1.93), '10': (6.74, 2.57)},
}

def parse_eval(result_dir):
    out = subprocess.run(
        ['python', 'kitti_eval/eval_odom.py',
         f'--result={result_dir}', '--align=7dof'],
        capture_output=True, text=True).stdout
    results = {}
    for seq in ['09', '10']:
        m = re.search(rf'{seq}.*?([0-9]+\.[0-9]+).*?([0-9]+\.[0-9]+)', out)
        results[seq] = (float(m.group(1)), float(m.group(2))) if m else (None, None)
    return results

experiments = [
    ('A  Offline VO',         'vo_results_offline'),
    ('B  Full online (paper)', 'vo_results_full'),
    ('C  LoRA-attn rank 8',    'vo_results_lora'),
    ('D  LoRA + trigger 0.8',  'vo_results_lora_trigger'),
]

W = 28
print(f"{'Condition':<{W}} | {'Seq 09':^19} | {'Seq 10':^19}")
print(f"{'':<{W}} | {'t_err%':>9} {'r°/100m':>8} | {'t_err%':>9} {'r°/100m':>8}")
print('-' * 72)
for label, key in [('  Paper offline', 'offline'), ('  Paper online', 'online')]:
    r09, r10 = PAPER[key]['09'], PAPER[key]['10']
    print(f"{label:<{W}} | {r09[0]:9.2f} {r09[1]:8.2f} | {r10[0]:9.2f} {r10[1]:8.2f}")
print('-' * 72)
for name, d in experiments:
    if not os.path.exists(d):
        continue
    r = parse_eval(d)
    def fmt(v): return f'{v:9.2f}' if v else '      N/A'
    t09, r09 = r.get('09', (None, None))
    t10, r10 = r.get('10', (None, None))
    print(f"{name:<{W}} | {fmt(t09)} {fmt(r09)} | {fmt(t10)} {fmt(r10)}")

## Trajectory Visualization

In [ ]:
# ── Cell 13: Trajectory Plots ────────────────────────────────────────────────
exec(open('scripts/colab_eval_viz.py').read())

In [ ]:
# ── Cell 14: Download All Results ────────────────────────────────────────────
import zipfile, os
from google.colab import files

with zipfile.ZipFile('/content/sgvo_lora_results.zip', 'w') as z:
    for d in ['vo_results_offline', 'vo_results_full',
              'vo_results_lora', 'vo_results_lora_trigger']:
        if not os.path.exists(d): continue
        for f in os.listdir(d):
            z.write(os.path.join(d, f))

files.download('/content/sgvo_lora_results.zip')

## vKITTI 2 — Cross-Domain Generalization (G3)
Scene01 (447 frames) & Scene20 (837 frames) under **clone / fog / morning**.
All conditions share identical GT trajectories (`kitti_eval/gt_poses_vkitti/`, in repo).
Condition D probes with the **ratio signal** (predicted-pose loss / zero-motion
loss): raw photometric loss is contrast-confounded — fog LOWERS it while pose
accuracy collapses, so the CUSUM trigger never fired on any vKITTI run. The
ratio cancels the contrast factor. Reference stats (mu=0.7314,
sigma=0.0660, measured on 2,788 unadapted KITTI 09/10 probe frames)
are passed explicitly because vKITTI sequences start already domain-shifted —
start-of-sequence calibration would hide the shift.


In [ ]:
# ── Cell 15: vKITTI 2 Data Setup (Scene01/20 × clone/fog/morning) ────────────
# Fast path: wget pre-built tars from GitHub Release vkitti-data-v1.
# Fallback (first time): download official vKITTI2 rgb tar (7.5 GB), extract
# the subset, build the tars, and offer vkitti_assets.zip for download so the
# assets can be uploaded to the release for future sessions.
import os, glob, subprocess, shutil

SCENES  = {'01': 'Scene01', '20': 'Scene20'}
CONDS   = ['clone', 'fog', 'morning', 'rain', 'sunset']
RELEASE = 'https://github.com/AreebaAzizRajput/SG-VO/releases/download/vkitti-data-v1'
VK_RGB  = 'https://download.europe.naverlabs.com/virtual_kitti_2.0.3/vkitti_2.0.3_rgb.tar'

def seq_dir(cond, seq): return f'data/vkitti/{cond}/sequences/{seq}/image_2'
def seq_ok(cond, seq):  return len(glob.glob(seq_dir(cond, seq) + '/*.jpg')) > 100

missing = [(c, s) for c in CONDS for s in SCENES if not seq_ok(c, s)]
built   = []

for cond, seq in list(missing):   # fast path
    tar = f'vkitti_{seq}_{cond}.tar'
    r = subprocess.run(['wget', '-q', f'{RELEASE}/{tar}', '-O', f'/content/{tar}'])
    if r.returncode == 0 and os.path.getsize(f'/content/{tar}') > 1e6:
        os.makedirs(f'data/vkitti/{cond}/sequences', exist_ok=True)
        subprocess.run(['tar', '-xf', f'/content/{tar}', '-C', f'data/vkitti/{cond}/sequences'], check=True)
        missing.remove((cond, seq))
        print(f'✅ {seq}/{cond} from release')
    if os.path.exists(f'/content/{tar}'): os.remove(f'/content/{tar}')

if missing:                        # fallback builder
    print(f'Building from official vKITTI2 tar for: {missing}')
    if not os.path.exists('/content/vkitti_rgb.tar'):
        subprocess.run(['wget', '-q', '--show-progress', VK_RGB, '-O', '/content/vkitti_rgb.tar'], check=True)
    os.makedirs('/content/vk_extract', exist_ok=True)
    subprocess.run(['tar', '-xf', '/content/vkitti_rgb.tar', '-C', '/content/vk_extract',
                    '--wildcards'] + [f'{SCENES[s]}/{c}/frames/rgb/Camera_0/*' for c, s in missing],
                   check=True)
    os.makedirs('/content/vk_assets', exist_ok=True)
    for cond, seq in missing:
        src = f'/content/vk_extract/{SCENES[seq]}/{cond}/frames/rgb/Camera_0'
        dst = seq_dir(cond, seq)
        os.makedirs(dst, exist_ok=True)
        for i, f in enumerate(sorted(os.listdir(src))):
            shutil.copy(f'{src}/{f}', f'{dst}/{i:06d}.jpg')
        subprocess.run(['tar', '-cf', f'/content/vk_assets/vkitti_{seq}_{cond}.tar',
                        '-C', f'data/vkitti/{cond}/sequences', seq], check=True)
        built.append((cond, seq))
        print(f'✅ built {seq}/{cond}')
    shutil.rmtree('/content/vk_extract', ignore_errors=True)
    if os.path.exists('/content/vkitti_rgb.tar'): os.remove('/content/vkitti_rgb.tar')

# intrinsics: vKITTI2 (1242x375) scaled to the 832x256 network input
fx, fy, cx, cy = 725.0087, 725.0087, 620.5, 187.0
sx, sy = 832/1242, 256/375
cam = f'{fx*sx:.6f} 0 {cx*sx:.6f}\n0 {fy*sy:.6f} {cy*sy:.6f}\n0 0 1\n'
for cond in CONDS:
    for seq in SCENES:
        if seq_ok(cond, seq):
            with open(seq_dir(cond, seq) + '/cam.txt', 'w') as f: f.write(cam)

for cond in CONDS:
    for seq in SCENES:
        n = len(glob.glob(seq_dir(cond, seq) + '/*.jpg'))
        print(f'  {cond}/{seq}: {n} images', '✅' if n > 100 else '❌')

if built:                          # hand tars over for release upload
    import zipfile
    from google.colab import files
    with zipfile.ZipFile('/content/vkitti_assets.zip', 'w') as z:
        for f in glob.glob('/content/vk_assets/*.tar'): z.write(f, os.path.basename(f))
    files.download('/content/vkitti_assets.zip')
    print('⬇️  vkitti_assets.zip — hand to the local agent for release upload')

In [ ]:
# ── Cell 16: vKITTI Experiments — A/B/C/D per weather condition ──────────────
# ~3.5 h total for all three conditions. Edit RUN_CONDS to split across
# sessions (results accumulate in separate vo_vkitti_* dirs).
RUN_CONDS   = ['clone', 'fog', 'morning']   # 'rain', 'sunset' also available
# CUSUM reference stats for the RATIO probe signal (predicted-pose loss /
# zero-motion loss — contrast-invariant, unlike the raw photometric loss that
# fog pushes DOWN). Measured on 2,788 unadapted KITTI 09/10 probe frames.
KREF_MU     = 0.7314
KREF_SIGMA  = 0.0660
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'

COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

import glob, os, zipfile

def cond_done(cond):
    # a condition is complete when all four methods produced both trajectories
    return all(os.path.exists(f'vo_vkitti_{cond}_{m}/{s}.txt')
               for m in ['offline', 'full', 'lora', 'trigger'] for s in ['01', '20'])

def persist(cond):
    # survive Colab disconnects: eval + download the moment a condition finishes
    for m in ['offline', 'full', 'lora', 'trigger']:
        d = f'vo_vkitti_{cond}_{m}/'
        if glob.glob(d + '*.txt'):
            print(f'=== eval {d} ===')
            !python kitti_eval/eval_odom_vkitti.py --result={d} \
                --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
    zf = f'/content/vkitti_{cond}_results.zip'
    with zipfile.ZipFile(zf, 'w') as z:
        for d in glob.glob(f'vo_vkitti_{cond}_*/'):
            for f in os.listdir(d):
                z.write(os.path.join(d, f))
    from google.colab import files
    files.download(zf)

for cond in RUN_CONDS:
    if cond_done(cond):
        print(f'✅ {cond} already complete — skipping')
        continue
    DD = f'data/vkitti/{cond}/sequences/'
    for seq in ['01', '20']:
        print(f'\n════ {cond} / seq {seq} ════')
        print('— A: offline —')
        !python test_vo.py --img-height 256 --img-width 832 --sequence {seq} \
            --pretrained-posenet {POSE_NET} --dataset-dir {DD} \
            --output-dir vo_vkitti_{cond}_offline/
        print('— B: full online —')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --output-dir vo_vkitti_{cond}_full/
        print('— C: LoRA always —')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --lora-rank 8 --lora-targets attention \
            --output-dir vo_vkitti_{cond}_lora/
        print('— D: LoRA + CUSUM (KITTI reference) —')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --lora-rank 8 --lora-targets attention \
            --cusum-h 16 --cusum-ref-mean {KREF_MU} --cusum-ref-std {KREF_SIGMA} \
            --probe-signal ratio \
            --output-dir vo_vkitti_{cond}_trigger/
    persist(cond)
print('\n✅ vKITTI experiment runs complete')

In [ ]:
# ── Cell 17: vKITTI Evaluation + Results Download ────────────────────────────
import glob, os, re, zipfile

for d in sorted(glob.glob('vo_vkitti_*/')):
    if glob.glob(d + '*.txt'):
        print(f'\n=== {d} ===')
        !python kitti_eval/eval_odom_vkitti.py --result={d} \
            --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20

# compact summary table from the result.txt files
print('\n' + '=' * 76)
print(f'{"condition":<12s} {"method":<10s} {"seq":<4s} {"t_err%":>8s} {"r_err":>8s} {"ATE":>8s}')
for d in sorted(glob.glob('vo_vkitti_*/')):
    rf = os.path.join(d, 'result.txt')
    if not os.path.exists(rf): continue
    m = re.match(r'vo_vkitti_(\w+?)_(offline|full|lora|triggerplus|trigger)', d)
    cond, method = m.groups() if m else ('?', '?')
    txt = open(rf).read()
    for seq, terr, rerr, ate in re.findall(
            r'Sequence:\s+(\d+)\s+Trans. err. \(%\):\s+([\d.]+)\s+'
            r'Rot. err. \(deg/100m\):\s+([\d.]+)\s+ATE \(m\):\s+([\d.]+)', txt):
        print(f'{cond:<12s} {method:<10s} {seq:<4s} {terr:>8s} {rerr:>8s} {ate:>8s}')

with zipfile.ZipFile('/content/sgvo_vkitti_results.zip', 'w') as z:
    for d in sorted(glob.glob('vo_vkitti_*/')):
        for f in glob.glob(d + '*.txt'):
            z.write(f)
from google.colab import files
files.download('/content/sgvo_vkitti_results.zip')

## D+ — Normalized objective + undoable adaptation (novelty ablation)
Two additions over D, both default-off so D's semantics are unchanged:
`--adapt-loss norm` divides the photometric objective by the frame's zero-motion
baseline (fog shrinks raw-loss gradients by the same transmission factor that
confounds the probe — the division makes adaptation step sizes contrast-invariant);
`--reset-on-recovery` probes the pristine base model (adapters disabled) while
shifted and, once it stays in-band for 30 consecutive frames, zeroes the adapters —
recovering the exact pre-adaptation model. `trigger_events_<seq>.txt` in the output
dir records every adapt/reset for the paper's timeline figure.

In [ ]:
# ── Cell 18: vKITTI D+ — normalized loss + reset-on-recovery ────────────────
# Run AFTER Cell 15 (data) and Cell 3 (checkpoints). Compare against
# vo_vkitti_<cond>_trigger (D) in Cell 17's table; results land in
# vo_vkitti_<cond>_triggerplus.
RUN_CONDS   = ['clone', 'fog', 'morning']   # 'rain', 'sunset' also available
KREF_MU     = 0.7314    # ratio-signal KITTI reference (2,788 frames)
KREF_SIGMA  = 0.0660
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'

COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

import glob, os, zipfile

for cond in RUN_CONDS:
    if all(os.path.exists(f'vo_vkitti_{cond}_triggerplus/{s}.txt') for s in ['01','20']):
        print(f'✅ D+ {cond} already complete — skipping')
        continue
    DD = f'data/vkitti/{cond}/sequences/'
    for seq in ['01', '20']:
        print(f'\n════ D+ {cond} / seq {seq} ════')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --lora-rank 8 --lora-targets attention \
            --cusum-h 16 --cusum-ref-mean {KREF_MU} --cusum-ref-std {KREF_SIGMA} \
            --probe-signal ratio --adapt-loss norm \
            --reset-on-recovery --recovery-patience 30 \
            --output-dir vo_vkitti_{cond}_triggerplus/
    d = f'vo_vkitti_{cond}_triggerplus/'
    print(f'=== eval {d} ===')
    !python kitti_eval/eval_odom_vkitti.py --result={d} \
        --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
    zf = f'/content/vkitti_{cond}_dplus_results.zip'
    with zipfile.ZipFile(zf, 'w') as z:
        for f in os.listdir(d):
            z.write(os.path.join(d, f))
    from google.colab import files
    files.download(zf)
print('\n✅ D+ runs complete')

## Weather-switch splices — the recovery centerpiece (clone → fog → clone)
All vKITTI conditions of a scene share the same camera trajectory, so frames can
be spliced by index and the GT poses stay exactly valid. This is the experiment
that shows the full loop on one physically consistent drive: CUSUM fires entering
the fog, LoRA adapts, and after the fog clears the recovery reset returns the
exact pristine model. Compare A (offline) vs D+ on the same splice; the event
timeline in `trigger_events_<seq>.txt` overlaid on the error curve is the paper's
centerpiece figure. Requires Cell 15 (clone + fog data) first.

In [ ]:
# ── Cell 19: Weather-switch splice — build + run A and D+ ───────────────────
SHIFT_COND  = 'fog'          # spliced into the middle third; 'rain'/'sunset'/'morning' also work
KREF_MU     = 0.7314         # ratio-signal KITTI reference (2,788 frames)
KREF_SIGMA  = 0.0660
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'

COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

SPLICE = f'splice{SHIFT_COND}'
for seq in ['01', '20']:
    !python scripts/make_weather_splice.py \
        --base-dir data/vkitti/clone/sequences \
        --shift-dir data/vkitti/{SHIFT_COND}/sequences \
        --out-dir data/vkitti/{SPLICE}/sequences \
        --sequence {seq} --shift-start 0.33 --shift-end 0.67

for seq in ['01', '20']:
    DD = f'data/vkitti/{SPLICE}/sequences/'
    print(f'\n════ splice {SHIFT_COND} / seq {seq} — A: offline ════')
    !python test_vo.py --img-height 256 --img-width 832 --sequence {seq} \
        --pretrained-posenet {POSE_NET} --dataset-dir {DD} \
        --output-dir vo_vkitti_{SPLICE}_offline/
    print(f'\n════ splice {SHIFT_COND} / seq {seq} — D+: trigger+undo ════')
    !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
        --lora-rank 8 --lora-targets attention \
        --cusum-h 16 --cusum-ref-mean {KREF_MU} --cusum-ref-std {KREF_SIGMA} \
        --probe-signal ratio --adapt-loss norm \
        --reset-on-recovery --recovery-patience 30 \
        --output-dir vo_vkitti_{SPLICE}_triggerplus/

print('\n=== event timelines ===')
!cat vo_vkitti_{SPLICE}_triggerplus/trigger_events_*.txt

import os, zipfile, glob
for d in [f'vo_vkitti_{SPLICE}_offline/', f'vo_vkitti_{SPLICE}_triggerplus/']:
    print(f'=== eval {d} ===')
    !python kitti_eval/eval_odom_vkitti.py --result={d} \
        --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
zf = f'/content/vkitti_{SPLICE}_results.zip'
with zipfile.ZipFile(zf, 'w') as z:
    for d in glob.glob(f'vo_vkitti_{SPLICE}_*/'):
        for f in os.listdir(d):
            z.write(os.path.join(d, f))
from google.colab import files
files.download(zf)

## Condition E — CUSUM-gated FULL adaptation (D's brain, B's muscles)

The vKITTI grid decomposed the system: the **trigger works** (fires 4–9% in-domain
vs 28–38% in fog) but **LoRA-on-pose is a null actuator** (`disp_net` is frozen
under LoRA, and fog corrupts depth — C moved fog seq 20 by 0.13% while adapting
every frame). E pairs the working detector with the working actuator: full
both-net adaptation, executed **only when CUSUM fires**.

Expected: ≈ A in clone/morning (trigger quiet), → B in fog (36.27 → toward 9.67),
at roughly a third of B's adaptation cost. This is the headline-table row.

In [ ]:
# ── Cell 20: Condition E — CUSUM-gated full adaptation ──────────────────────
# No LoRA: --lora-rank 0 (default) => adapting trains BOTH nets in full,
# but only on frames where the ratio-signal CUSUM fires.
RUN_CONDS  = ['clone', 'fog', 'morning']
KREF_MU    = 0.7314   # KITTI training-domain ratio stats (2,788 probe frames)
KREF_SIGMA = 0.0660
POSE_NET   = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET   = 'checkpoints/dispnet112_model_best.pth.tar'
COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

import glob, os, zipfile

def e_done(cond):
    return all(os.path.exists(f'vo_vkitti_{cond}_gatedfull/{s}.txt') for s in ['01', '20'])

def persist_e(cond):
    d = f'vo_vkitti_{cond}_gatedfull/'
    if glob.glob(d + '*.txt'):
        print(f'=== eval {d} ===')
        !python kitti_eval/eval_odom_vkitti.py --result={d} \
            --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
    zf = f'/content/vkitti_{cond}_gatedfull_results.zip'
    with zipfile.ZipFile(zf, 'w') as z:
        for f in os.listdir(d):
            z.write(os.path.join(d, f))
    from google.colab import files
    files.download(zf)

for cond in RUN_CONDS:
    if e_done(cond):
        print(f'✅ {cond} E already complete — skipping')
        continue
    DD = f'data/vkitti/{cond}/sequences/'
    for seq in ['01', '20']:
        print(f'\n════ E: gated full / {cond} / seq {seq} ════')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --cusum-h 16 --cusum-ref-mean {KREF_MU} --cusum-ref-std {KREF_SIGMA} \
            --probe-signal ratio \
            --output-dir vo_vkitti_{cond}_gatedfull/
    persist_e(cond)
print('\n✅ Condition E runs complete')

## Network decomposition — WHERE does the fog rescue live?

B (full online) trains `disp_net` and `pose_net` jointly, so its 3.7× fog rescue
does not say which network absorbs the shift. These runs adapt exactly **one** net
in full (the other truly frozen: grads off **and** BatchNorm pinned), ungated —
adapt every frame like B — to isolate the actuator question from the trigger.

- **B_pose** is the capacity upper bound for *any* pose-side PEFT: if it fails
  under fog, LoRA-on-pose was doomed at any rank and C/D's no-op is fully
  explained. If it works, C's failure was a rank/targets problem.
- **B_disp** localizes the depth contribution — if it alone recovers the rescue,
  "gated LoRA-on-depth" restores the original cheap-adaptation narrative.

Fog answers the rescue question; clone answers where B's in-domain *damage*
(r_err 1.15 → 2.87) comes from. Needs repo at commit `03b714e`+ (`--adapt-nets`).

In [ ]:
# ── Cell 21: Decomposition — B_pose / B_disp on fog + clone ──────────────────
RUN_CONDS = ['fog', 'clone']   # fog: locate the rescue; clone: locate the damage
POSE_NET  = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET  = 'checkpoints/dispnet112_model_best.pth.tar'
COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

import glob, os, zipfile

def decomp_done(cond):
    return all(os.path.exists(f'vo_vkitti_{cond}_b{m}/{s}.txt')
               for m in ['pose', 'disp'] for s in ['01', '20'])

def persist_decomp(cond):
    for m in ['pose', 'disp']:
        d = f'vo_vkitti_{cond}_b{m}/'
        if glob.glob(d + '*.txt'):
            print(f'=== eval {d} ===')
            !python kitti_eval/eval_odom_vkitti.py --result={d} \
                --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
    zf = f'/content/vkitti_{cond}_decomp_results.zip'
    with zipfile.ZipFile(zf, 'w') as z:
        for d in glob.glob(f'vo_vkitti_{cond}_b*/'):
            for f in os.listdir(d):
                z.write(os.path.join(d, f))
    from google.colab import files
    files.download(zf)

for cond in RUN_CONDS:
    if decomp_done(cond):
        print(f'✅ {cond} decomposition already complete — skipping')
        continue
    DD = f'data/vkitti/{cond}/sequences/'
    for seq in ['01', '20']:
        for mode in ['pose', 'disp']:
            print(f'\n════ B_{mode} / {cond} / seq {seq} ════')
            !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
                --adapt-nets {mode} \
                --output-dir vo_vkitti_{cond}_b{mode}/
    persist_decomp(cond)
print('\n✅ Decomposition runs complete')

## Joint LoRA — the PEFT configuration the mechanism permits

The decomposition proved the fog rescue is **irreducibly joint**: the trajectory
is a pure function of `pose_net` (disp-only adaptation is bit-identical to
offline), and pose-only adaptation is catastrophic (fog seq 01: 41.2% vs
offline's 9.96%) because its gradients flow through frozen fog-corrupted depth.

`--lora-nets both` puts adapters on **both** nets: pose cross-attention (21.5K)
+ all 14 depth-decoder convs (48.9K) = 70,432 params, 0.117% of both nets.
Depth adapters keep the loss landscape honest while pose adapters move.

- **C_joint** (ungated): does joint rank-8 capacity recover any of B's fog
  rescue (36.27 → 9.67)? This is the capacity-ladder rung between B and the
  dead pose-only LoRA.
- **D_joint** (CUSUM-gated): the deployable form, if C_joint works.

Needs repo at commit `a88c66e`+.

In [ ]:
# ── Cell 22: Joint LoRA — C_joint (always) + D_joint (CUSUM) on fog + clone ──
RUN_CONDS  = ['fog', 'clone']   # fog: does joint capacity rescue? clone: damage check
KREF_MU    = 0.7314
KREF_SIGMA = 0.0660
POSE_NET   = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET   = 'checkpoints/dispnet112_model_best.pth.tar'
COMMON = (f'-p 1 -s 0.1 -c 0.5 --img-height 256 --img-width 832 '
          f'--pretrained-posenet {POSE_NET} --pretrained-disp {DISP_NET} '
          f'--epochs 2 --lr 1e-4 --sequence-length 3 --resnet-layers 50 '
          f'--select-best 1 --with-mask 1 --with-auto-mask 1 --padding-mode border')

import glob, os, zipfile

def joint_done(cond):
    return all(os.path.exists(f'vo_vkitti_{cond}_{m}/{s}.txt')
               for m in ['jointlora', 'jointtrigger'] for s in ['01', '20'])

def persist_joint(cond):
    for m in ['jointlora', 'jointtrigger']:
        d = f'vo_vkitti_{cond}_{m}/'
        if glob.glob(d + '*.txt'):
            print(f'=== eval {d} ===')
            !python kitti_eval/eval_odom_vkitti.py --result={d} \
                --gt=kitti_eval/gt_poses_vkitti/ --align=7dof --seqs 1 20
    zf = f'/content/vkitti_{cond}_joint_results.zip'
    with zipfile.ZipFile(zf, 'w') as z:
        for d in glob.glob(f'vo_vkitti_{cond}_joint*/'):
            for f in os.listdir(d):
                z.write(os.path.join(d, f))
    from google.colab import files
    files.download(zf)

for cond in RUN_CONDS:
    if joint_done(cond):
        print(f'✅ {cond} joint LoRA already complete — skipping')
        continue
    DD = f'data/vkitti/{cond}/sequences/'
    for seq in ['01', '20']:
        print(f'\n════ C_joint: LoRA both nets, always / {cond} / seq {seq} ════')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --lora-rank 8 --lora-nets both --lora-targets attention \
            --output-dir vo_vkitti_{cond}_jointlora/
        print(f'\n════ D_joint: LoRA both nets + CUSUM / {cond} / seq {seq} ════')
        !python test_vo_online.py {COMMON} --sequence {seq} --dataset-dir {DD} \
            --lora-rank 8 --lora-nets both --lora-targets attention \
            --cusum-h 16 --cusum-ref-mean {KREF_MU} --cusum-ref-std {KREF_SIGMA} \
            --probe-signal ratio \
            --output-dir vo_vkitti_{cond}_jointtrigger/
    persist_joint(cond)
print('\n✅ Joint LoRA runs complete')